# Training Notebook: `C_relation_entity_markers` on the Main Trainable Dataset

This notebook provides a reproducibility entry point for retraining the relation-plus-entity-marked verifier on the released main trainable dataset. It can be run either locally from this repository or in Google Colab.

The notebook saves the trained checkpoint under `models/main_trainable_dataset/C/models/` and the associated metrics under `models/main_trainable_dataset/C/results/training/`. It illustrates one retained verifier configuration rather than the full model comparison reported in the paper.


## 1. Environment setup

This section checks whether the notebook is running in Colab and installs the Python dependencies required for training. In a local repository checkout, install the dependencies first with `pip install -r requirements.txt`.


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print('IN_COLAB:', IN_COLAB)

if IN_COLAB:
    !pip -q install transformers accelerate scikit-learn seaborn


## 2. Repository and data setup

This section locates the released repository workspace and the published dataset files used for retraining. In Colab, a zip export of the repository can be uploaded. In a local checkout, the notebook uses the repository structure directly.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import sys
import zipfile

USE_DATA_ZIP = IN_COLAB
ZIP_PATH = Path('/content/github_repo_workspace.zip')
ZIP_EXTRACT_DIR = Path('/content/github_repo')


def find_workspace() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts').exists() and (candidate / 'datasets').exists() and (candidate / 'ontology').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root containing scripts, datasets, and ontology.')


if USE_DATA_ZIP:
    from google.colab import files

    if ZIP_EXTRACT_DIR.exists():
        shutil.rmtree(ZIP_EXTRACT_DIR)
    ZIP_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    if ZIP_PATH.exists():
        ZIP_PATH.unlink()

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No zip uploaded.')
    uploaded_name = next(iter(uploaded.keys()))
    print('Uploaded:', uploaded_name)
    Path(uploaded_name).rename(ZIP_PATH)

    with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
        archive.extractall(ZIP_EXTRACT_DIR)

    extracted_roots = [item for item in ZIP_EXTRACT_DIR.iterdir() if item.is_dir()]
    ROOT = extracted_roots[0] if len(extracted_roots) == 1 else ZIP_EXTRACT_DIR
else:
    ROOT = find_workspace()

if str(ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(ROOT / 'scripts'))

print('ROOT:', ROOT)



## 3. Training configuration

This section loads the training module and defines the configuration used for the `C_relation_entity_markers` verifier on the released main trainable dataset.


In [ ]:
import importlib
import pandas as pd
import torch

import train_ontology_format_verifier as trainer
importlib.reload(trainer)

trainer.ROOT = ROOT
trainer.DATA_ROOT = ROOT / 'datasets' / 'main_trainable_dataset'
trainer.GROUPED_DATA_ROOT = ROOT / 'datasets' / 'original_verifier_dataset'
trainer.MODEL_ROOT = ROOT / 'models'
trainer.RESULTS_ROOT = ROOT / 'results' / 'training'
trainer.MODEL_ROOT.mkdir(parents=True, exist_ok=True)
trainer.RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

FORMAT_NAME = 'C_relation_entity_markers'
FORMAT_CODE = 'C'
DATASET_NAME = 'main_trainable_dataset'
ACTIVE_DATA_ROOT = ROOT / 'datasets' / DATASET_NAME
BENCHMARK_ROOT = ROOT / 'datasets' / 'evaluation_only_interview_grounded_dataset'

MODEL_NAME = 'bert-base-uncased'
EPOCHS = 4
BATCH_SIZE = 32
MAX_LEN = 256
MODEL_SUFFIX = '_main_trainable_dataset'

ARCHIVE_ROOT = ROOT / 'models' / DATASET_NAME / FORMAT_CODE
MODEL_STORE_DIR = ARCHIVE_ROOT / 'models'
RESULTS_STORE_DIR = ARCHIVE_ROOT / 'results' / 'training'
MODEL_STORE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_STORE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_OUT = MODEL_STORE_DIR / f'{FORMAT_NAME}{MODEL_SUFFIX}'
METRICS_OUT = RESULTS_STORE_DIR / f'{FORMAT_NAME}{MODEL_SUFFIX}_metrics.json'

print('ACTIVE_DATA_ROOT:', ACTIVE_DATA_ROOT)
print('BENCHMARK_ROOT:', BENCHMARK_ROOT)
print('ARCHIVE_ROOT:', ARCHIVE_ROOT)
print('MODEL_OUT:', MODEL_OUT)
print('METRICS_OUT:', METRICS_OUT)


## 4. Inspect dataset splits

This section loads the released train, validation, and test splits and provides a brief inspection of their size and label distribution.


In [ ]:
train_rows, val_rows, test_rows = trainer.load_format_rows(FORMAT_NAME, data_root=ACTIVE_DATA_ROOT)

for split_name, rows in [('train', train_rows), ('val', val_rows), ('test', test_rows)]:
    df = pd.DataFrame(rows)
    print(split_name, len(rows))
    print('  labels:', df['label'].value_counts().to_dict())
    print('  sources:', df['source_family'].value_counts().to_dict() if 'source_family' in df else 'MISSING')

display(pd.DataFrame(train_rows[:5])[['text', 'label', 'source_family', 'source_dataset']])


## 5. Train the verifier

This section runs training for the relation-plus-entity-marked verifier and stores the resulting checkpoint in the local archived-model layout used by this repository.


In [ ]:
metrics = trainer.train_format(
    format_name=FORMAT_NAME,
    model_name=MODEL_NAME,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN,
    data_root=ACTIVE_DATA_ROOT,
    output_model_dir=MODEL_OUT,
    output_metrics_path=METRICS_OUT,
)

print(json.dumps({
    'format': metrics['format'],
    'model_name': metrics['model_name'],
    'epochs': metrics['epochs'],
    'batch_size': metrics['batch_size'],
    'max_len': metrics['max_len'],
    'train': metrics['train'],
    'val': metrics['val'],
    'test': metrics['test'],
}, indent=2))

print('Saved model to:', MODEL_OUT)
print('Saved metrics to:', METRICS_OUT)


## Optional benchmark evaluation

The released evaluation-only interview-grounded dataset provides an additional generalization check outside the main training split. This optional step evaluates the freshly trained verifier on that benchmark.


## 6. Optional evaluation on the released interview-grounded benchmark

This section evaluates the trained verifier on the released evaluation-only interview-grounded dataset.


In [ ]:
from end_to_end_verifier_evaluation import compute_binary_metrics, predict_texts


def load_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]


if BENCHMARK_ROOT.exists():
    benchmark_path = BENCHMARK_ROOT / FORMAT_NAME / 'eval.jsonl'
    rows = load_jsonl(benchmark_path)
    preds = predict_texts(MODEL_OUT, [row['text'] for row in rows], batch_size=32, max_len=MAX_LEN)
    benchmark_metrics = compute_binary_metrics([row['label'] for row in rows], [pred['prediction'] for pred in preds])
    benchmark_out = RESULTS_STORE_DIR / f'{FORMAT_NAME}{MODEL_SUFFIX}_natural_benchmark_eval.json'
    benchmark_out.write_text(json.dumps({
        'format': FORMAT_NAME,
        'model_dir': str(MODEL_OUT),
        'benchmark_path': str(benchmark_path),
        **benchmark_metrics,
    }, indent=2), encoding='utf-8')
    print(json.dumps(benchmark_metrics, indent=2))
    print('Saved benchmark metrics:', benchmark_out)
else:
    print('Benchmark root missing, skipped:', BENCHMARK_ROOT)


## 7. Export results

This section packages the trained checkpoint and metrics for later inspection or download.


In [ ]:
import zipfile

zip_out = RESULTS_STORE_DIR / f'{FORMAT_NAME}{MODEL_SUFFIX}_model_metrics.zip'
files_to_zip = []
if MODEL_OUT.exists():
    files_to_zip.extend([p for p in MODEL_OUT.rglob('*') if p.is_file()])
files_to_zip.extend([p for p in RESULTS_STORE_DIR.glob(f'{FORMAT_NAME}{MODEL_SUFFIX}*.json') if p.is_file()])

with zipfile.ZipFile(zip_out, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in files_to_zip:
        archive.write(file_path, arcname=file_path.relative_to(ROOT))

print('Wrote:', zip_out)
print('Files:', len(files_to_zip))

try:
    from google.colab import files
    files.download(str(zip_out))
except Exception as exc:
    print('Download skipped outside Colab:', exc)
